In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

In [ ]:
model = ChatOllama(model = "llama3.1", temperature=0)
model

In [ ]:
@tool
def make_payment(name : str, amount : float) -> str:
    """Make a payment of {amount} to {name}"""

    return f"Made a payment of {amount} to {name}"

In [ ]:
middleware = HumanInTheLoopMiddleware(
    interrupt_on = {
        "make_payment" : True
    }
)
checkpointer = InMemorySaver()

In [ ]:
agent = create_agent(
    model = model,
    tools = [make_payment],
    middleware = [middleware],
    checkpointer = checkpointer
)
agent

In [ ]:
config = {
    "configurable" : {
        "thread_id" : "test-123"
    }
}
agent.invoke({
    "messages" : [
        {
            "role" : "user",
            "content" : "Transfer 500 to Leo"
        }
    ], 
}, 
config = config
)

In [ ]:
result = agent.invoke(
    Command(
        resume = {
            "decisions" : [
                {
                    "type" : "approve",                    
                }
            ]
        }
    ), 
    config = config
)

In [ ]:
result